# Greek Banking Sector — Executive Summary

**Scope:** the four listed Greek banking groups, FY2022–FY2024: Alpha Services and Holdings (Alpha Bank), Eurobank Ergasias Services and Holdings (Eurobank), National Bank of Greece (NBG), Piraeus Financial Holdings (Piraeus).
**Data:** consolidated statements in the FY2023 and FY2024 annual reports and regulatory ratios from the FY2022–FY2024 reports, every figure traced to a PDF page (`02_Banking_Sector_Dashboard/data/processed/sources.csv`).
**Analyst:** Spyros Papastergiou | papastergiousp@gmail.com

---

This notebook brings together notebooks 01–05:
1. A one-page **KPI scorecard** for FY2024
2. **What the numbers say** about each bank
3. A **risk matrix** for the sector after the rate peak

In [1]:
import sqlite3
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

BANKS  = ['Eurobank', 'Alpha Bank', 'Piraeus Bank', 'NBG']
COLORS = {'Eurobank': '#0067B1', 'Alpha Bank': '#E2231A',
           'Piraeus Bank': '#F7A600', 'NBG': '#003087'}

con  = sqlite3.connect('../02_Banking_Sector_Dashboard/data/greek_banking_final.db')
kpis = pd.read_sql('SELECT * FROM kpis ORDER BY bank, year', con)
con.close()

k24 = kpis[kpis.year == 2024].set_index('bank')
print(f'Loaded {len(kpis)} rows — {kpis.bank.nunique()} banks x {kpis.year.nunique()} years')

Loaded 12 rows — 4 banks x 3 years


## 1. One-Page KPI Scorecard (FY2024)

In [2]:
# ── Scorecard table ───────────────────────────────────────────────────────────
cols = ['roe', 'roa', 'nim', 'cost_to_income', 'cet1', 'loan_to_deposit', 'net_profit', 'total_assets']
labels = ['ROE (%)', 'ROA (%)', 'NIM (%)', 'C/I (%)', 'CET1 (%)', 'L/D (%)', 'Net Profit (€m)', 'Assets (€bn)']

score = k24.loc[BANKS, cols].copy()
score['total_assets'] = score['total_assets'] / 1000
score.columns = labels

# Format
fmt = {
    'ROE (%)': '{:.1f}', 'ROA (%)': '{:.2f}', 'NIM (%)': '{:.2f}',
    'C/I (%)': '{:.1f}', 'CET1 (%)': '{:.1f}', 'L/D (%)': '{:.1f}',
    'Net Profit (€m)': '{:,.0f}', 'Assets (€bn)': '{:.1f}'
}

def best(col):
    """Return index of best value per metric (direction-aware)."""
    lower_better = {'C/I (%)': True, 'L/D (%)': True}
    return score[col].idxmin() if lower_better.get(col) else score[col].idxmax()

print('FY2024 PEER SCORECARD')
print('=' * 80)
print(f'{"":20}' + ''.join(f'{b:>15}' for b in BANKS))
print('-' * 80)
for lbl, col in zip(labels, cols):
    row_data = score[lbl]
    winner = best(lbl)
    row_str = f'{lbl:20}'
    for b in BANKS:
        val = fmt[lbl].format(row_data[b])
        flag = ' ★' if b == winner else '  '
        row_str += f'{val + flag:>15}'
    print(row_str)
print('=' * 80)
print('★ = best in class for that metric')

FY2024 PEER SCORECARD
                           Eurobank     Alpha Bank   Piraeus Bank            NBG
--------------------------------------------------------------------------------
ROE (%)                      17.2 ★          8.5           13.7           14.4  
ROA (%)                      1.60 ★         0.91           1.36           1.55  
NIM (%)                      2.77           2.28           2.67           3.15 ★
C/I (%)                      32.9           39.0           31.8 ★         36.5  
CET1 (%)                     16.8           16.3           14.5           18.3 ★
L/D (%)                      64.8           76.5           64.8           62.7 ★
Net Profit (€m)             1,448 ★          654          1,066          1,158  
Assets (€bn)                101.2 ★         72.1           80.0           75.0  
★ = best in class for that metric


## 2. What the numbers say

### 2.1 Rates drove 2023, volumes drove 2024
Sector NII rose 51% in 2023 (€5.4bn → €8.1bn) and only 6% in 2024 (€8.6bn). In the rate/volume walk (notebook 04), the margin effect explains all of the 2023 increase, while in 2024 about half of the gain came from balance sheet growth, mostly Eurobank's consolidation of Hellenic Bank. With the ECB deposit rate at 2.00% by mid-2025, margin is now a headwind and loan growth and fees have to carry earnings.

### 2.2 NBG: highest margin and capital, most rate-sensitive
NBG has the highest NIM (3.15% of average assets), the highest CET1 ratio (18.3%) and the lowest loan-to-deposit ratio (62.7%). Its margin rose the most as rates went up (from 1.69% in 2022), and the link works in reverse too: NBG has the highest NIM sensitivity to the ECB rate of the four (about 0.40pp of NIM per 1pp of rate, notebook 01 in 04_forecasting), so it has the most margin to lose as rates fall. Its fee share of income is the lowest of the four (14.8%).

### 2.3 Eurobank: best returns, bigger after Hellenic Bank
Eurobank has the highest ROE (17.2%) and ROA (1.60%). With Hellenic Bank, total assets rose 27% and deposits 37% in 2024, which also lowered its loan-to-deposit ratio from 72% to 65%. Its DTC share of CET1 is the lowest (36%), so its capital is the highest quality of the four.

### 2.4 Piraeus: lowest costs, but 2024 profit leans on low provisions
Piraeus has the best cost-to-income ratio (31.8%) and the highest fee share (20.3%). Net profit rose 35% in 2024 mainly because loan impairments fell from €404m to €181m (cost of risk from 1.08% to 0.46%). At the 2023 cost of risk, 2024 net profit would be about 16% lower. Its CET1 ratio (14.5%) is the lowest of the four and DTCs are 63% of its CET1 capital.

### 2.5 Alpha Bank: the laggard, catching up on capital
Alpha has the lowest ROE (8.5%), the highest cost-to-income ratio (39.0%), the highest cost of risk (0.96%) and NPE ratio (3.8%), and the highest loan-to-deposit ratio (76.5%). Capital improved the most: CET1 rose from 13.2% to 16.3% between 2022 and 2024. In the simplified stress test (notebook 02 in 04_forecasting) Alpha is the only bank that would make a loss, which matches its weakest result in the official EBA 2023 stress test.

### 2.6 2022 is not a clean base year
2022 profits include one-offs at all four banks (notebook 05): trading gains at Eurobank, a disposal gain at Piraeus, discontinued operations at NBG and disposal gains at Alpha. On reported figures sector net profit fell 3.5% in 2023; on underlying figures it rose by about 42%.

## 3. NIM Compression Scenario (2025–2026)

In [3]:
# NIM sensitivity to ECB cuts (assumption-driven)
# ECB deposit rate: 3.00% at end-2024, 2.00% from June 2025.
# Pass-through of the rate cut to NIM is an assumption: 25% (deposit costs fall too) or 50%.
cut_pp = 3.00 - 2.00
nim_2024 = k24['nim']

sens = pd.DataFrame({
    'NIM 2024 (%)': nim_2024,
    'NIM, 25% pass-through': nim_2024 - cut_pp * 0.25,
    'NIM, 50% pass-through': nim_2024 - cut_pp * 0.50,
}).loc[BANKS].round(2)
sens['NII impact at 50% (€m)'] = (-cut_pp * 0.50 / 100 * k24.loc[BANKS, 'avg_total_assets']).round(0)

print('NIM sensitivity to a 100bp ECB cut (assumption-driven)')
print(sens.to_string())
print('\nNBG starts from the highest NIM; in euro terms the impact scales with average assets, so Eurobank is hit the most.')

NIM sensitivity to a 100bp ECB cut (assumption-driven)
              NIM 2024 (%)  NIM, 25% pass-through  NIM, 50% pass-through  NII impact at 50% (€m)
bank                                                                                            
Eurobank              2.77                   2.52                   2.27                  -452.0
Alpha Bank            2.28                   2.03                   1.78                  -361.0
Piraeus Bank          2.67                   2.42                   2.17                  -391.0
NBG                   3.15                   2.90                   2.65                  -374.0

NBG starts from the highest NIM; in euro terms the impact scales with average assets, so Eurobank is hit the most.


## 4. Sector Risk Matrix

In [4]:
k = k24.loc[BANKS]
risks = [
    ('ECB rate cuts → NIM compression', 'High', 'High',
     f"NBG has the highest NIM ({k.loc['NBG','nim']:.2f}%)"),
    ('Credit cost normalisation', 'Medium', 'Medium',
     f"Piraeus cost of risk {k.loc['Piraeus Bank','cost_of_risk']:.2f}% in 2024 vs 1.08% in 2023"),
    ('Loan growth slowdown', 'Medium', 'Medium', 'Volume growth has to offset the margin decline'),
    ('Capital quality (DTC)', 'Low', 'Medium',
     f"DTCs are {k.loc['Piraeus Bank','dtc_to_cet1']:.0f}% of Piraeus CET1 capital vs {k.loc['Eurobank','dtc_to_cet1']:.0f}% at Eurobank"),
    ('Macro or sovereign shock', 'Low', 'High', 'DTCs are a claim on the Greek State; banks also hold Greek government bonds'),
    ('Integration risk', 'Medium', 'Medium', 'Eurobank integrating Hellenic Bank'),
]

print(f'{"Risk factor":40} {"Likelihood":11} {"Impact":8} {"Key exposure"}')
print('-' * 110)
for r in risks:
    print(f'{r[0]:40} {r[1]:11} {r[2]:8} {r[3]}')

print('\nMitigants (FY2024 data)')
print(f"  CET1 ratios between {k['cet1'].min():.1f}% and {k['cet1'].max():.1f}%")
print(f"  NPE ratios between {k['npe_ratio'].min():.1f}% and {k['npe_ratio'].max():.1f}%, down from 5.2% to 7.8% in 2022")
print(f"  Cost-to-income between {k['cost_to_income'].min():.1f}% and {k['cost_to_income'].max():.1f}%")

Risk factor                              Likelihood  Impact   Key exposure
--------------------------------------------------------------------------------------------------------------
ECB rate cuts → NIM compression          High        High     NBG has the highest NIM (3.15%)
Credit cost normalisation                Medium      Medium   Piraeus cost of risk 0.46% in 2024 vs 1.08% in 2023
Loan growth slowdown                     Medium      Medium   Volume growth has to offset the margin decline
Capital quality (DTC)                    Low         Medium   DTCs are 63% of Piraeus CET1 capital vs 36% at Eurobank
Macro or sovereign shock                 Low         High     DTCs are a claim on the Greek State; banks also hold Greek government bonds
Integration risk                         Medium      Medium   Eurobank integrating Hellenic Bank

Mitigants (FY2024 data)
  CET1 ratios between 14.5% and 18.3%
  NPE ratios between 2.6% and 3.8%, down from 5.2% to 7.8% in 2022
  Cost-to-inco

## 5. ROE vs CET1 — Positioning Chart

In [5]:
fig = go.Figure()

for bank in BANKS:
    row = k24.loc[bank]
    fig.add_trace(go.Scatter(
        x=[row['cet1']], y=[row['roe']],
        mode='markers+text',
        name=bank,
        text=[bank.replace(' Bank', '')],
        textposition='top center',
        textfont=dict(size=11, color=COLORS[bank]),
        marker=dict(
            size=row['total_assets'] / 1000,
            sizemode='area',
            color=COLORS[bank],
            opacity=0.75,
            line=dict(width=1.5, color='white')
        ),
        hovertemplate=(
            f'<b>{bank}</b><br>'
            'CET1: %{x:.1f}%<br>'
            'ROE: %{y:.1f}%<br>'
            f'Assets: €{row["total_assets"]/1000:.0f}bn'
            '<extra></extra>'
        )
    ))

fig.add_vline(x=k24['cet1'].mean(), line_dash='dot', line_color='rgba(255,255,255,0.15)')
fig.add_hline(y=k24['roe'].mean(), line_dash='dot', line_color='rgba(255,255,255,0.15)')

for txt, x, y, anchor in [
    ('High ROE\nHigh Capital', 18.5, 18.0, 'left'),
    ('Low ROE\nHigh Capital',  18.5,  9.5, 'left'),
    ('High ROE\nLow Capital',  13.2, 18.0, 'right'),
    ('Low ROE\nLow Capital',   13.2,  9.5, 'right'),
]:
    fig.add_annotation(x=x, y=y, text=txt, showarrow=False,
                       font=dict(size=8.5, color='rgba(255,255,255,0.18)'),
                       xanchor=anchor)

fig.update_layout(
    title=dict(text='ROE vs CET1 — FY2024 (bubble = assets)', font=dict(size=13), x=0.5),
    paper_bgcolor='#141921', plot_bgcolor='#141921',
    font=dict(family='DM Sans, sans-serif', color='#8892a8', size=11),
    xaxis=dict(title='CET1 Ratio (%)', gridcolor='rgba(255,255,255,0.06)', zeroline=False),
    yaxis=dict(title='ROE (%)', gridcolor='rgba(255,255,255,0.06)', zeroline=False, ticksuffix='%'),
    showlegend=False,
    width=640, height=420,
    margin=dict(l=60, r=40, t=50, b=60)
)
fig.show()
try:
    fig.write_image('../02_Banking_Sector_Dashboard/outputs/roe_cet1_matrix.png', scale=2, width=640, height=420)
except Exception:
    pass

## 6. Sector 3-Year KPI Evolution

In [6]:
metrics = [
    ('roe',            'ROE (%)',       '%'),
    ('nim',            'NIM (%)',       '%'),
    ('cost_to_income', 'C/I Ratio (%)', '%'),
    ('cet1',           'CET1 (%)',      '%'),
]

fig = make_subplots(rows=2, cols=2, subplot_titles=[m[1] for m in metrics],
                    vertical_spacing=0.18, horizontal_spacing=0.12)

for idx, (col, lbl, sfx) in enumerate(metrics):
    r, c = divmod(idx, 2)
    for bank in BANKS:
        bdata = kpis[kpis.bank == bank].sort_values('year')
        fig.add_trace(go.Scatter(
            x=bdata['year'], y=bdata[col],
            name=bank, legendgroup=bank,
            showlegend=(idx == 0),
            mode='lines+markers',
            line=dict(color=COLORS[bank], width=2.2),
            marker=dict(size=6, color=COLORS[bank]),
            hovertemplate=f'{bank}<br>{lbl}: %{{y:.2f}}{sfx}<extra></extra>'
        ), row=r+1, col=c+1)
    fig.update_xaxes(tickvals=[2022, 2023, 2024], row=r+1, col=c+1,
                     gridcolor='rgba(255,255,255,0.06)', zeroline=False)
    fig.update_yaxes(ticksuffix=sfx, row=r+1, col=c+1,
                     gridcolor='rgba(255,255,255,0.06)', zeroline=False)

fig.update_layout(
    paper_bgcolor='#141921', plot_bgcolor='#141921',
    font=dict(family='DM Sans, sans-serif', color='#8892a8', size=10.5),
    title=dict(text='Greek Banks — Key Metrics 2022–2024', font=dict(size=13), x=0.5),
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.08, font=dict(size=9.5)),
    width=820, height=520,
    margin=dict(l=55, r=30, t=70, b=60)
)
fig.show()
try:
    fig.write_image('../02_Banking_Sector_Dashboard/outputs/sector_kpi_trends.png', scale=2, width=820, height=520)
except Exception:
    pass

In [7]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(kpis) == 12, 'Expected 12 rows (4 banks x 3 years)'
assert kpis.bank.nunique() == 4 and kpis.year.nunique() == 3

nii_22 = kpis[kpis.year == 2022]['nii'].sum()
nii_24 = kpis[kpis.year == 2024]['nii'].sum()
assert nii_24 > nii_22 * 1.4, f'Expected sector NII to rise >40% 2022-2024, got {nii_24/nii_22-1:.1%}'
assert k24['cet1'].idxmax() == 'NBG'
assert k24['cost_to_income'].idxmin() == 'Piraeus Bank'
assert k24['roe'].idxmax() == 'Eurobank'

print('All checks passed.')
print(f'   Sector NII 2022: €{nii_22:,.0f}m  →  2024: €{nii_24:,.0f}m  (+{nii_24/nii_22-1:.1%})')
print(f'   Best ROE 2024:   {k24["roe"].idxmax()} ({k24["roe"].max():.1f}%)')
print(f'   Best ROA 2024:   {k24["roa"].idxmax()} ({k24["roa"].max():.2f}%)')
print(f'   Best CET1 2024:  {k24["cet1"].idxmax()} ({k24["cet1"].max():.1f}%)')
print(f'   Best C/I 2024:   {k24["cost_to_income"].idxmin()} ({k24["cost_to_income"].min():.1f}%)')

All checks passed.
   Sector NII 2022: €5,376m  →  2024: €8,596m  (+59.9%)
   Best ROE 2024:   Eurobank (17.2%)
   Best ROA 2024:   Eurobank (1.60%)
   Best CET1 2024:  NBG (18.3%)
   Best C/I 2024:   Piraeus Bank (31.8%)
